# FSDP 原理与 TorchTitan 框架

---

## 1. 从单进程训练到数据并行

Qwen3-1.7B 有 17 亿参数，每个 BF16 参数占 2 字节：

$$\text{模型权重} = 1.7\text{B} \times 2 \text{ bytes} = 3.4 \text{ GB} \approx 3.17 \text{ GiB}$$

这里的 3.17 GiB 只包含模型权重。训练还需要梯度、优化器状态、激活和临时缓冲区；它们随 dtype、优化器、batch、序列长度和 activation checkpoint 配置变化，应以当前配置的实测峰值为准。

本课程的 Qwen3-1.7B 可以放入单个可见 die，因此学习 FSDP 不是为了解决当前模型无法加载的问题，而是为了理解两件事：多个 rank 如何共同完成一次数据并行更新，以及模型状态如何分片，从而扩展到更大模型、更长序列或更大 batch。下面先建立两 rank 拓扑，再比较 DDP 与 FSDP，最后说明 TorchTitan 如何把这些能力接入完整训练流程。参见 [TorchTitan 功能列表](https://github.com/pytorch/torchtitan#key-features-available)和 PyTorch [`fully_shard` 文档](https://docs.pytorch.org/docs/stable/distributed.fsdp.fully_shard.html)。

### 1.1 本课程的两 rank 拓扑

- **die**：A3 环境暴露的计算设备。本课程主线看到 2 个 die，每个约 64 GB。
- **进程 / rank**：`torchrun --nproc_per_node=2` 启动 2 个训练进程，编号为 rank 0 和 rank 1，通常每个进程绑定一个 die。
- **world size**：参与默认 process group 的总进程数，本课程单机主线为 2。
- **process group**：collective 的参与者集合。
- **collective**：组内所有 rank 按相同顺序共同参与的通信，例如 all-gather、reduce-scatter。

这些概念描述了谁参与计算和通信。接下来先看每个 rank 保存完整模型的 DDP，再看分片保存模型状态的 FSDP。

### 1.2 DDP：每个 rank 保存完整副本

数据并行把一个 global batch 分成多个 local batch，交给不同 rank 并行计算。PyTorch DistributedDataParallel（DDP）通常为每个设备启动一个进程；每个 rank 保存完整的参数、梯度和优化器状态，并对自己的 local batch 执行 forward/backward。反向传播期间，各 rank 对梯度执行 all-reduce，随后使用相同的归约结果执行 optimizer step，因此模型副本保持一致。DataLoader 或 sampler 负责让各 rank 读取不同输入。

```text
rank 0：完整模型副本 + local batch 0 ── forward/backward ─┐
                                                           ├─ gradient all-reduce → 各 rank optimizer.step()
rank 1：完整模型副本 + local batch 1 ── forward/backward ─┘
```

DDP 的执行模型直观，也能通过梯度 bucket 让 all-reduce 与 backward 重叠；代价是数据并行规模增加时，每个 rank 仍然保存完整模型状态，单 rank 的模型状态显存不会随 rank 数下降。参见 PyTorch [DDP 文档](https://docs.pytorch.org/docs/stable/generated/torch.nn.parallel.DistributedDataParallel.html)。

---

## 2. FSDP：分片保存模型状态

FSDP 不是另一种训练语义：各 rank 仍处理不同数据，并共同形成一次参数更新。它改变的是模型状态在 rank 间的存放方式。以 FULL_SHARD 为例，参数、梯度和优化器状态都分布在数据并行 rank 上；计算某个 FSDP unit 时，才临时收集该 unit 所需的完整参数。

| 对比项 | DDP | FULL_SHARD |
|---|---|---|
| 参数、梯度、优化器状态 | 每个 rank 保存完整副本 | 在数据并行 rank 间分片 |
| Forward 前 | 参数已完整驻留，无须参数 all-gather | all-gather 当前 FSDP unit 的参数 |
| Backward 中 | 梯度 all-reduce，各 rank 得到完整梯度 | reduce-scatter 梯度，各 rank 只保留自己的 shard |
| 显存特征 | 模型状态显存不随 rank 数下降 | 稳态只保存 shard，但会出现当前 unit 的临时未分片参数 |

### 2.1 Forward：all-gather 当前 unit 的参数

![FSDP-forward](./images/fsdp-forward.png)

每个 rank 使用收集后的参数计算自己的 local batch。计算结束后是否立即 reshard 由 `reshard_after_forward` 等配置决定；随后进入下一个 FSDP unit。这里的未分片参数只属于当前 unit，不代表整套模型同时展开。

### 2.2 Backward：按需重新收集参数并 reduce-scatter 梯度

![FSDP-backward](./images/fsdp-backward.png)

如果参数在 forward 后已经 reshard，backward 计算该 unit 前还需要再次 unshard（通常对应一次 all-gather）。得到各 rank 的本地梯度贡献后，reduce-scatter 同时完成两件事：先归约梯度，再把结果按 shard 分给对应 rank。每个 rank 随后只更新自己持有的参数和优化器状态 shard。

### 2.3 FSDP unit 与通信重叠

一次 `fully_shard(module)` 会管理该 module 中尚未由嵌套 FSDP unit 管理的参数。Transformer 通常按从子层到根模块的顺序调用 `fully_shard`：计算当前 unit 时只 all-gather 当前参数组，其他 unit 仍保持分片。多个 unit 可以降低同时未分片的参数峰值，并为参数预取与当前计算重叠创造条件；但 unit 太大时峰值显存增加，太小时 collective 数量和调度开销增加。因此，unit 边界需要结合模型结构、显存和 profile 选择。

### 2.4 FSDP1 与 FSDP2

| 维度 | FSDP1 | FSDP2 (`fully_shard`) |
|------|-------|-----------------------|
| 参数表示 | 将一组参数 flatten/concat 为 `FlatParameter` 后分片 | 可分片参数分别用 dim-0 sharded `DTensor` 表示 |
| API 形式 | `FullyShardedDataParallel(module)` wrapper | composable `fully_shard(module)`，在原 module 上注册 hooks 和 FSDP 行为 |
| 参数名称 | wrapper/state-dict 逻辑负责映射原参数 | 原始 fully-qualified names 保持不变 |
| unit 划分 | 可按 module 包装 | 按 module 自底向上调用；一次调用形成一个通信组 |
| 调度控制 | 提供 prefetch 等选项 | 额外提供 `unshard()`、`reshard()` 和显式 prefetch 控制 |

两者都能按 module 划分 unit，也都可能重叠通信与计算。FSDP2 的主要变化是 per-parameter DTensor 表示和 composable API；unit 边界仍由 module 结构与 `fully_shard` 的调用位置决定。详细差异见 TorchTitan 官方 [FSDP1 → FSDP2 说明](https://github.com/pytorch/torchtitan/blob/main/docs/fsdp.md)以及 PyTorch [FSDP1 文档](https://docs.pytorch.org/docs/stable/fsdp.html)。实际重叠效果取决于 unit 边界、预取策略、模型和硬件。

---

## 3. TorchTitan：把 PyTorch 原生训练能力组织成可配置流程

FSDP 只解决模型状态的分片与相关通信。完整的大模型训练还要协调数据、其他并行维度、activation checkpoint、优化器、checkpoint 和日志。TorchTitan 是 PyTorch 原生训练平台，不只是 FSDP wrapper；它把这些组件接入统一的配置和训练生命周期，最终仍可通过一条命令启动。

![](./images/training-startup-flow.png)

TorchTitan 主要组织以下能力：

- **分布式训练**：按配置组合 DDP/FSDP2、张量并行（TP）、上下文并行（CP）或流水线并行（PP），把数据、模型状态和计算映射到多个进程
- **数据加载**：对话数据的 tokenization、label masking、batch 拼接
- **激活管理**：选择 activation checkpoint 策略，在显存与重计算之间权衡
- **检查点管理**：训练中断恢复，以及通过适配器加载或导出 Hugging Face 权重
- **混合精度**：配置参数、计算和通信使用的 dtype，在精度、显存与性能之间权衡
- **优化器与学习率调度**：Adam 状态管理、warmup/decay
- **可观测性**：记录 loss、grad_norm、显存、吞吐量、TFLOPs 和 MFU，并按需采集 profile

Python config registry 提供可复用的基准配置，CLI override 只覆盖本次运行需要改变的字段，两者共同构成最终训练配置。

### 3.1 启动流程

> 源码：`torchtitan_npu/entry.py`

```text
scripts/run_train.sh → torchrun → torchtitan_npu.entry
                     → 解析 module、config 与 CLI override
                     → 构建 Trainer、设备 mesh 与并行维度
                     → 构建数据与模型
                     → 执行 converter 和并行化
                     → 构建 optimizer、scheduler 与 checkpoint manager
                     → train loop
```

### 3.2 训练脚本

> 源码：`scripts/run_train.sh`


```bash
NGPU=2 \
MODULE=torchtitan_npu.models.qwen3 \
CONFIG=sft_qwen3_1_7b_wordle \
bash scripts/run_train.sh
```


| 参数 | 含义 |
|------|------|
| `NGPU=2` | 在 2 个可见 die 上启动 2 个进程（world size 2） |
| `MODULE=torchtitan_npu.models.qwen3` | 使用 Qwen3 的 model_registry |
| `CONFIG=sft_qwen3_1_7b_wordle` | 使用 Wordle SFT 配置 |

---

## 4. model_registry：训练配置

> 源码：`torchtitan_npu/models/qwen3/__init__.py`

在 torchtitan 中，每个训练任务都需要这样的配置：


```python
def model_registry(flavor: str):
    spec = _upstream_model_registry(flavor)
    return spec.__class__(
        name=spec.name,                              # 模型名称
        flavor=spec.flavor,                          # 规格（1.7B / 0.6B / debugmodel）
        model=spec.model,                            # 模型类（nn.Module）
        parallelize_fn=parallelize_qwen3,            # 模型相关并行化总入口
        pipelining_fn=spec.pipelining_fn,            # 流水线并行（本章不用）
        build_loss_fn=spec.build_loss_fn,            # loss 函数
        post_optimizer_build_fn=spec.post_optimizer_build_fn,
        state_dict_adapter=Qwen3StateDictAdapter,    # HF 格式权重加载
    )
```


| 字段 | 作用 |
|------|------|
| `model` | 模型结构定义（Qwen3 的 Transformer） |
| `parallelize_fn` | 模型相关并行化总入口；Qwen3 先检查/应用 CP 约束，再委托上游处理 TP、compile、FSDP 或 DDP 等路径 |
| `state_dict_adapter` | 加载/保存 Hugging Face 格式权重 |
| `flavor` | `"1.7B"` → 1.7B 参数版本；`"0.6B"` 是 28 层、约 0.6B 参数的正式 flavor，不等于 8 层 `debugmodel` |
| `build_loss_fn` | 训练用的 loss 函数 |

配置中 `model_spec=model_registry("1.7B")` 选择模型规格及其构建、并行化、loss 和权重适配入口；具体启用哪种并行仍由最终 `parallelism` 配置决定。

---

## 小结

- **DDP** 让每个 rank 保存完整模型副本、处理不同数据，并用 gradient all-reduce 保持更新一致。
- **FSDP** 延续数据并行语义，同时在 rank 间分片模型状态；当前 FSDP unit 计算前按需 all-gather，梯度通过 reduce-scatter 完成归约与分片。
- **FSDP2** 主要采用 composable `fully_shard` 与 per-parameter DTensor sharding；FSDP1 和 FSDP2 都可重叠通信与计算，实际效果以 profile 为准。
- **TorchTitan** 把多维并行、数据处理、activation checkpoint、检查点、混合精度和可观测性组织成可配置训练流程。
- **model_registry** 注册模型，`parallelize_fn` 是模型相关并行化总入口，`state_dict_adapter` 处理权重加载。

ModelConverter 机制将在第 4 章详细讲解——它允许你无侵入地替换模型中的算子。

## 练习

1. （判断题）FULL_SHARD 在 forward 前通过 all-gather 收集当前 FSDP unit 的参数，并在 backward 中通过 reduce-scatter 归约并分片梯度。

2. （单选题）`data_parallel_replicate_degree=1`、`data_parallel_shard_degree=2`、`CP=1` 时采用哪种并行方式？
    A. DDP，FSDP degree 1
    B. FSDP，FSDP mesh degree 2
    C. TP，TP degree 2
    D. PP，PP degree 2

3. （单选题）`model_registry("1.7B")` 中 `parallelize_fn=parallelize_qwen3` 的作用是什么？
    A. 下载数据集
    B. 根据并行配置进入 Qwen3 的 CP、TP、FSDP/DDP 等路径
    C. 转换模型为 ONNX 格式
    D. 启动推理服务

4. （多选题）FSDP2 的主要特点包括哪些？
    A. composable `fully_shard` API
    B. per-parameter DTensor sharding
    C. 所有参数永久保持未分片
    D. 只能把整套模型作为一个 FSDP unit

In [ ]:
!cat ./answer/02.02_answer.txt